In [142]:
import numpy as np

## A. Implement compute_homo

In [143]:
def compute_homo(matched_corner_pairs):
    A = np.zeros((2 * len(matched_corner_pairs), 9))
    
    for i, (pt1, pt2) in enumerate(matched_corner_pairs):
        x1, y1 = pt1
        x2, y2 = pt2
        
        A[2*i] = [0, 0, 0, x1, y1, 1, -y2*x1, -y2*y1, -y2]
        A[2*i + 1] = [x1, y1, 1, 0, 0, 0, -x2*x1, -x2*y1, -x2]
        
    print("Matrix A:")
    print(A)
    
    _U,_D,Vt = np.linalg.svd(A) 
    h = Vt[-1]
    H = h.reshape(3, 3)
    return H

## B. Testing if the implementation is correct

### B.1. The A matrix in 773-2026 Assignment 1 Phase 3 (Theory) - Question 1


In [144]:
# # Theory
# # S1 = {(1, 1, 1), (2, 3, 1), (4, 2, 1), (3, 5, 1))}
# # S2 = {(3, 4, 1), (5, 8, 1), (9, 6, 1), (7, 12, 1)},
theory_pairs = np.array(
    [
        [[1, 1], [3, 4]],
        [[2, 3], [5, 8]],
        [[4, 2], [9, 6]],
        [[3, 5], [7, 12]],
    ]
)
theory_H = compute_homo(theory_pairs)

print(theory_H)

Matrix A:
[[  0.   0.   0.   1.   1.   1.  -4.  -4.  -4.]
 [  1.   1.   1.   0.   0.   0.  -3.  -3.  -3.]
 [  0.   0.   0.   2.   3.   1. -16. -24.  -8.]
 [  2.   3.   1.   0.   0.   0. -10. -15.  -5.]
 [  0.   0.   0.   4.   2.   1. -24. -12.  -6.]
 [  4.   2.   1.   0.   0.   0. -36. -18.  -9.]
 [  0.   0.   0.   3.   5.   1. -36. -60. -12.]
 [  3.   5.   1.   0.   0.   0. -21. -35.  -7.]]
[[ 0.84152255 -0.22410859 -0.0274559 ]
 [ 0.29881145  0.24389965  0.24389965]
 [ 0.04980191 -0.02490095  0.17175173]]


#### The `A` matrix matches but not the `H` as below

![Alt text|100](assets/assigment_1_3_theory_q1.png)

### B.2. The H matrix in Tutorial

In [145]:
# # Tutorial
# # S1 = {(2, 2, 1), (2, 4, 1), (6, 4, 1), (6, 2, 1))}
# # S2 = {(-2, -2, 1), (-1, -4, 1), (-6, -1, 1), (-6, -5, 1)},
tutorial_pairs = np.array(
    [
        [[2, 2], [-2, -2]],
        [[2, 4], [-1, -4]],
        [[6, 4], [-6, -1]],
        [[6, 2], [-6, -5]],
    ]
)

tutorial_H = compute_homo(tutorial_pairs)

print(tutorial_H)

Matrix A:
[[ 0.  0.  0.  2.  2.  1.  4.  4.  2.]
 [ 2.  2.  1.  0.  0.  0.  4.  4.  2.]
 [ 0.  0.  0.  2.  4.  1.  8. 16.  4.]
 [ 2.  4.  1.  0.  0.  0.  2.  4.  1.]
 [ 0.  0.  0.  6.  4.  1.  6.  4.  1.]
 [ 6.  4.  1.  0.  0.  0. 36. 24.  6.]
 [ 0.  0.  0.  6.  2.  1. 30. 10.  5.]
 [ 6.  2.  1.  0.  0.  0. 36. 12.  6.]]
[[ 0.16006408  0.09369604 -0.81984039]
 [ 0.14640007 -0.09369604 -0.4177282 ]
 [-0.05270403 -0.01561601  0.29280014]]


#### Both `A` matrix and `H` match the slide

![](assets/assigment_1_3_tutorial_slide.png)

## C. Debuging


The S1 sets contains:
- s1_1: (1, 1)
- s1_2: (2, 3)
- s1_4: (3, 5)

All are on the line $y = 2x - 1$ => `collinearity`

I am douting this issue impact the calculation. Cuz it look like there are infinite transformation.

When I run on different system the result can be different. The issue also stared in the slide.

![image.png](assets/assigment_1_3_hints_slide.png)


I guess because of the `LAPACK` package

On the [numpy.linalg.svd](https://numpy.org/devdocs/reference/generated/numpy.linalg.svd.html), it said

> The decomposition is performed using LAPACK routine _gesdd.

On the [LAPACK](https://www.netlib.org/lapack/#:~:text=The%20original%20goal%20of%20the,Linear%20Algebra%20Subprograms%20(BLAS).)

> We use the term "transportable" instead of "portable" because, for fastest possible performance, LAPACK requires that highly optimized block matrix operations be already implemented on each machine.

Previous result is from Apple silicon processor, while the image below is from intel core i7 system.

![](assets/assigment_1_3_intel_windows.jpeg)

We can also validate the $Ah=0$

In [146]:
A = np.array(
    [
        [0, 0, 0, 1, 1, 1, -4, -4, -4],
        [1, 1, 1, 0, 0, 0, -3, -3, -3],
        [0, 0, 0, 2, 3, 1, -16, -24, -8],
        [2, 3, 1, 0, 0, 0, -10, -15, -5],
        [0, 0, 0, 4, 2, 1, -24, -12, -6],
        [4, 2, 1, 0, 0, 0, -36, -18, -9],
        [0, 0, 0, 3, 5, 1, -36, -60, -12],
        [3, 5, 1, 0, 0, 0, -21, -35, -7],
    ]
)

# from the image
intel_H = np.array(
    [
        [-0.04614434, 0.22572712, 0.42838206],
        [-0.30096949, 0.55579463, 0.55579463],
        [-0.05016158, 0.02508079, 0.22773574],
    ]
)


print(np.dot(A, intel_H.flatten()))
print(np.dot(A, theory_H.flatten()))

print(np.linalg.norm(np.dot(A, intel_H.flatten())))
print(np.linalg.norm(np.dot(A, theory_H.flatten())))

[-2.99999999e-08 -1.00000000e-08 -5.99999999e-08 -1.00000001e-08
 -7.00000000e-08 -5.99999999e-08 -9.00000007e-08 -9.99999991e-09]
[-1.55431223e-15 -2.77555756e-16 -6.66133815e-16 -5.55111512e-17
 -4.10782519e-15 -2.38697950e-15 -4.66293670e-15 -2.05391260e-15]
1.4628738866107983e-07
7.17448075233818e-15


The result is approximately 0